# xBD — Data Split

**Author:** Artur Zavistovskyi · **Project:** Mapping Natural-Disaster Damage from Satellite Imagery
**Source:** section 5 of `artur_xbd_NESH_CAN_RUN.ipynb`, split out and rewritten for a
**local, Kaggle-free** run.

Run [`01_eda.ipynb`](01_eda.ipynb) first — it writes the cached label scan this notebook reads.
If the cache is absent, this notebook rebuilds it itself (slower, but standalone).

---

## What this notebook does

`splits.csv → coverage audit → cleaning rules → leakage checks → eligible pool → stratified scene sample`

Output: `results/02_data_split/selected_scenes.csv`, the scene list that patch extraction consumes, plus the
per-split and per-event composition tables that justify it.

## Running it

Same requirements and layout as `01_eda.ipynb`:

```
pip install numpy pandas matplotlib pillow pyarrow notebook
```

Start Jupyter from the project root, or set `PROJECT_ROOT_OVERRIDE` in the PATHS cell.

## 1 · Setup

In [ ]:
# ---------------------------------------------------------------- dependency check
# Neither of these two notebooks needs torch: EDA and the split are pure pandas/PIL work.
# Fail here with an actionable message rather than with an ImportError halfway down.
import importlib.util

REQUIRED = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
            "PIL": "pillow", "pyarrow": "pyarrow"}          # pyarrow: the .parquet caches

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ImportError("Missing packages: " + ", ".join(missing) + "\n\n"
                      "    pip install " + " ".join(missing))
print("all required packages present")

In [ ]:
# ---------------------------------------------------------------- imports
import os, sys, json, re, time, random, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

warnings.filterwarnings("ignore", category=UserWarning)
ImageFile.LOAD_TRUNCATED_IMAGES = False   # we WANT truncated files to raise, so we can count them
Image.MAX_IMAGE_PIXELS = None

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__)

In [ ]:
# ================================================================ CONFIGURATION
# Everything tunable lives here. Nothing below this cell hard-codes a magic number.

# --- labels: the xBD 4-level joint damage scale -------------------------------
DAMAGE_CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]
NAME_TO_IDX    = {n: i for i, n in enumerate(DAMAGE_CLASSES)}
NUM_CLASSES    = len(DAMAGE_CLASSES)
SHORT          = ["none", "minor", "major", "destr"]      # compact axis labels

# "un-classified" is a 5th value annotators used when a building could NOT be assessed.
# It is not a damage level, so it is excluded from training - but it is our single best
# occlusion signal, so we keep and analyse it in section 4.3.
UNCLASSIFIED   = "un-classified"

# --- the four xBD origin directories (xbd/train/ is a duplicate of tier1, excluded) ---
ORIGINS = ("hold", "test", "tier1", "tier3")
SPLITS  = ("train", "val", "test_id", "test_ood")

# The five held-out wildfire events, frozen here so plots stay correct even when part of
# the dataset is not on disk yet (see scripts/prepare_splits.py on Bhuvanesh-branch).
FIRE_EVENTS = {"socal-fire", "santa-rosa-wildfire",
               "woolsey-fire", "pinery-bushfire", "portugal-wildfire"}

# --- patch geometry (set by EDA 3.4, consumed by the model notebooks) ---------
PATCH_SIZE   = 64     # px. Justified against the measured bbox distribution in section 3.4.
BBOX_PAD     = 10     # px of context kept around each building footprint
MIN_BBOX_PX  = 8      # cleaning rule: drop footprints smaller than this (section 4.5)
MIN_BUILDINGS_PER_SCENE = 5    # scene-sampling filter, calibrated in section 5.2

# Scene budget per split - how many scenes the model notebooks are allowed to extract from.
SCENE_BUDGET = {"train": 800, "val": 200, "test_id": 300, "test_ood": 400}

# --- diagnostics --------------------------------------------------------------
QC_SAMPLE_SCENES = 400    # scenes used for pixel-level image QC and duplicate hashing
OCCLUSION_TOP_N  = 12     # occlusion candidates displayed for manual review
SHOW_EXAMPLE_GRIDS = True # section 3.7 galleries; set False if the disk is slow (cosmetic only)

# --- I/O ----------------------------------------------------------------------
# PIL releases the GIL while decoding, so threads help even for a pure-Python loop.
# On a spinning disk, lower this: concurrent random reads make seek time worse.
IO_WORKERS = 16

# --- reproducibility ----------------------------------------------------------
SEED = 42            # same seed as prepare_splits.py, so anything re-derived here matches

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)

set_seed()
print("patch", PATCH_SIZE, "| seed", SEED, "| scene budget", SCENE_BUDGET)

In [ ]:
# ================================================================ PATHS (local run)
# The imagery layout in use here - the Kaggle xbd-dataset archive extracted under data_same/:
#
#   <project root>/
#       data/
#           splits.csv                     <- the team split (scripts/prepare_splits.py)
#       data_same/
#           xbd/{hold,test,tier1,tier3}/{images,labels,masks}
#
# Two other layouts are accepted without editing anything, because the resolver below searches
# several roots: xbd/ directly in the project root (docs/local_setup.md), and the per-archive
# form data/{train,test,tier3,hold}/ where `train` is the xView2 tar that IS xBD "tier1".

PROJECT_ROOT_OVERRIDE = None    # e.g. r"D:\UNI\DLSS\satellite-disaster-damage-mapping"


def find_project_root():
    """Look for data/splits.csv - in the override, then cwd, then its parents."""
    cands = [Path(PROJECT_ROOT_OVERRIDE)] if PROJECT_ROOT_OVERRIDE else []
    here = Path.cwd().resolve()
    cands += [here, *here.parents]
    for d in cands:
        if (d / "data" / "splits.csv").is_file():
            return d
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "data/splits.csv not found.\n"
        "Expected a directory containing data/splits.csv (the team split index).\n"
        "Either start Jupyter from the project root, or set PROJECT_ROOT_OVERRIDE above.\n"
        f"Searched upwards from: {Path.cwd()}")

DATA       = PROJECT_ROOT / "data"          # the split index lives here, not the imagery
SPLITS_CSV = DATA / "splits.csv"

# Roots searched for the imagery, in priority order. data_same/ holds the complete Kaggle
# archive and wins over anything left in data/ from an earlier partial download.
IMAGE_ROOTS = [PROJECT_ROOT / "data_same", DATA, PROJECT_ROOT]

WORK    = PROJECT_ROOT / "work";           WORK.mkdir(exist_ok=True)
CACHE   = WORK / "cache";                  CACHE.mkdir(parents=True, exist_ok=True)
RESULTS = PROJECT_ROOT / "results" / "02_data_split"; RESULTS.mkdir(parents=True, exist_ok=True)

print("cwd          :", Path.cwd())
print("project root :", PROJECT_ROOT)
print("splits.csv   :", SPLITS_CSV, f"({SPLITS_CSV.stat().st_size / 1e6:.1f} MB)")
print("cache        :", CACHE)
print("figures/CSVs :", RESULTS)

In [ ]:
# ---------------------------------------------------------------- resolve the origin directories
# splits.csv records paths as xbd\<origin>\<subdir>\<name>. We never use that string directly:
# we rebuild the path from (origin, subdir, name) so ANY on-disk layout works as long as each
# origin directory can be located.
#
# Two ordering rules matter here:
#   - roots are the OUTER loop, so a complete data_same/ wins over leftovers in data/;
#   - "xbd/tier1" is tried before "xbd/train", because the Kaggle mirror ships a partial
#     xbd/train/ that duplicates tier1 and is absent from splits.csv. Requiring BOTH images/
#     and labels/ rejects it a second time - it has no labels/ - but order alone is enough.

ORIGIN_DIR_CANDIDATES = {
    "tier1": ["xbd/tier1", "tier1", "train", "xbd/train"],   # the xView2 "train" tar IS tier1
    "test":  ["xbd/test",  "test"],
    "tier3": ["xbd/tier3", "tier3"],
    "hold":  ["xbd/hold",  "hold"],
}

ORIGIN_ROOT = {}
for origin, cands in ORIGIN_DIR_CANDIDATES.items():
    for base in IMAGE_ROOTS:
        for c in cands:
            p = base / c
            if (p / "images").is_dir() and (p / "labels").is_dir():
                ORIGIN_ROOT[origin] = p
                break
        if origin in ORIGIN_ROOT:
            break

print("origin directories found on disk:")
for origin in ORIGINS:
    root = ORIGIN_ROOT.get(origin)
    if root is None:
        print(f"  {origin:6s} -> MISSING")
    else:
        n_img = sum(1 for _ in (root / "images").glob("*.png"))
        print(f"  {origin:6s} -> {root}   ({n_img:,} image files)")

if not ORIGIN_ROOT:
    raise FileNotFoundError(
        "No xBD origin directory found. Searched these roots:\n"
        + "".join(f"  {r}\n" for r in IMAGE_ROOTS) +
        "Extract the xView2 imagery into one of them, as either xbd/{hold,test,tier1,tier3}/\n"
        "or {hold,test,tier3,train}/ - each with images/ and labels/ inside.")

## 2 · Rebuild the scene table and its coverage

In [ ]:
# ---------------------------------------------------------------- splits.csv -> one row per scene
splits_raw = pd.read_csv(SPLITS_CSV)
print("splits.csv:", splits_raw.shape)
display(splits_raw.head(3))

# Keep images + labels, drop masks: masks/ holds SEGMENTATION targets, which this per-building
# CLASSIFICATION task does not use. (The local tars call that directory `targets` - same thing,
# and equally unused, which is why the name mismatch does not matter.)
keep = splits_raw[splits_raw.subdir.isin(["images", "labels"])].copy()
keep["abs_path"] = [
    str(ORIGIN_ROOT[o] / s / n) if o in ORIGIN_ROOT else ""
    for o, s, n in zip(keep.origin, keep.subdir, keep["name"])
]
keep["col"] = keep.subdir.map({"images": "img", "labels": "lab"}) + "_" + keep.kind

scenes = (keep.pivot_table(index=["scene_id", "disaster", "origin", "split"],
                           columns="col", values="abs_path", aggfunc="first")
              .reset_index())
scenes.columns.name = None
scenes = scenes.rename(columns={"img_pre_disaster": "img_pre", "img_post_disaster": "img_post",
                                "lab_pre_disaster": "lab_pre", "lab_post_disaster": "lab_post"})
scenes["is_fire"] = scenes.disaster.isin(FIRE_EVENTS)

assert scenes.scene_id.is_unique, "scene_id must be unique after the pivot"
assert set(scenes.loc[scenes.split == "test_ood", "disaster"]) <= FIRE_EVENTS, \
    "test_ood must contain wildfire events only"

print(f"\nsplits.csv describes {len(scenes):,} scenes across {scenes.disaster.nunique()} events")
display(pd.crosstab(scenes.origin, scenes.split).reindex(index=ORIGINS, columns=SPLITS,
                                                         fill_value=0))

In [ ]:
# ---------------------------------------------------------------- what is ACTUALLY on disk
# splits.csv describes the full 11,034-scene dataset. This local machine may only hold part of
# it. A scene is usable only if pre image + post image + post label all exist; anything else is
# dropped here, once, loudly - rather than failing 200 cells later inside a thread pool.

NEEDED = ["img_pre", "img_post", "lab_post"]

t0 = time.time()
with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:
    present = {c: np.fromiter(ex.map(os.path.exists, scenes[c]), bool, len(scenes))
               for c in NEEDED}
scenes["files_ok"] = np.logical_and.reduce(list(present.values()))
print(f"checked {3 * len(scenes):,} paths in {time.time() - t0:.1f}s\n")

cov = (scenes.groupby(["origin", "split"]).files_ok.agg(on_disk="sum", in_splits_csv="size")
             .reset_index())
cov["coverage_%"] = (100 * cov.on_disk / cov.in_splits_csv).round(1)
print("scene coverage, origin x split:")
display(cov)

by_split = scenes.groupby("split").files_ok.agg(on_disk="sum", in_splits_csv="size").reindex(SPLITS)
by_split["coverage_%"] = (100 * by_split.on_disk / by_split.in_splits_csv).round(1)
print("scene coverage per split:")
display(by_split)

FULL_DATASET = bool(scenes.files_ok.all())
scenes_all, scenes = scenes, scenes[scenes.files_ok].reset_index(drop=True)

if FULL_DATASET:
    print("\nComplete dataset on disk - results are directly comparable to the reference run.")
else:
    lost = [o for o in ORIGINS if o not in ORIGIN_ROOT
            or not scenes_all.loc[scenes_all.origin == o, "files_ok"].any()]
    print(f"\n!! PARTIAL DATASET: {len(scenes):,} of {len(scenes_all):,} scenes "
          f"({100 * len(scenes) / len(scenes_all):.1f}%) are on disk.")
    print(f"   Missing or empty origins: {lost}")
    print("   Every number below is computed on what IS present and is therefore NOT")
    print("   comparable to the reference full-dataset run. Add the missing archives and")
    print("   re-run to reproduce it - the cache is keyed on coverage, so nothing is stale.")

if scenes.empty:
    raise RuntimeError("No usable scenes on disk. Extract the xView2 archives first.")

# Cache files are keyed on coverage: adding tier3/ or hold/ later changes the key, so a partial
# scan can never be silently reused as if it were the full one.
COVERAGE_TAG = "-".join(sorted(ORIGIN_ROOT)) + f"_{len(scenes)}"
print("\ncoverage tag (cache key):", COVERAGE_TAG)

fire_events = sorted(set(scenes.disaster) & FIRE_EVENTS)
print("wildfire events present   :", fire_events)
leak = scenes[scenes.split.isin(["train", "val"]) & scenes.disaster.isin(FIRE_EVENTS)]
assert len(leak) == 0, "wildfire leaked into the training pool"
print("fire scenes in train/val  :", len(leak), "(must be 0)")

In [ ]:
_WKT_NUM = re.compile(r"-?\d+\.?\d*")


def wkt_bbox(wkt: str):
    """Axis-aligned pixel bbox of a WKT polygon. Numbers alternate x, y."""
    nums = [float(v) for v in _WKT_NUM.findall(wkt)]
    xs, ys = nums[0::2], nums[1::2]
    return min(xs), min(ys), max(xs), max(ys)


def read_post_label(row):
    """Parse one post-disaster JSON -> (per-building records, per-scene metadata record)."""
    try:
        d = json.loads(Path(row.lab_post).read_text())
    except Exception as e:                    # counted as a corrupted label in section 4.2
        return [], {"scene_id": row.scene_id, "label_error": type(e).__name__,
                    "n_buildings": 0, "n_unclassified": 0}

    meta, feats = d.get("metadata", {}), d["features"].get("xy", [])
    recs, n_unc = [], 0
    for f in feats:
        props = f.get("properties", {})
        sub = props.get("subtype", UNCLASSIFIED)
        n_unc += (sub == UNCLASSIFIED)
        minx, miny, maxx, maxy = wkt_bbox(f["wkt"])
        recs.append((row.scene_id, props.get("uid", ""), sub, minx, miny, maxx, maxy))

    scene_rec = {
        "scene_id": row.scene_id, "label_error": None,
        "n_buildings": len(feats), "n_unclassified": n_unc,
        "gsd": meta.get("gsd"), "off_nadir_angle": meta.get("off_nadir_angle"),
        "sun_elevation": meta.get("sun_elevation"), "capture_date": meta.get("capture_date"),
        "disaster_type": meta.get("disaster_type"),
        "img_width": meta.get("width"), "img_height": meta.get("height"),
    }
    return recs, scene_rec

In [ ]:
# ---------------------------------------------------------------- full label scan (cached)
BLD_CACHE = CACHE / f"buildings__{COVERAGE_TAG}.parquet"
SCN_CACHE = CACHE / f"scene_meta__{COVERAGE_TAG}.parquet"

if BLD_CACHE.exists() and SCN_CACHE.exists():
    buildings  = pd.read_parquet(BLD_CACHE)
    scene_meta = pd.read_parquet(SCN_CACHE)
    print("loaded label scan from cache:", BLD_CACHE.name)
else:
    t0 = time.time()
    rows = list(scenes.itertuples(index=False))
    with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:      # I/O bound: many small JSON reads
        out = list(ex.map(read_post_label, rows))
    buildings = pd.DataFrame([r for recs, _ in out for r in recs],
                             columns=["scene_id", "uid", "subtype",
                                      "minx", "miny", "maxx", "maxy"])
    scene_meta = pd.DataFrame([s for _, s in out])
    buildings.to_parquet(BLD_CACHE, index=False)
    scene_meta.to_parquet(SCN_CACHE, index=False)
    print(f"scanned {len(rows):,} label files in {time.time() - t0:.0f}s")

# attach split / disaster / origin to every row and derive bbox geometry
buildings = buildings.merge(scenes[["scene_id", "disaster", "origin", "split"]],
                            on="scene_id", how="left")
buildings["bw"]     = buildings.maxx - buildings.minx
buildings["bh"]     = buildings.maxy - buildings.miny
buildings["barea"]  = buildings.bw * buildings.bh
buildings["aspect"] = buildings.bw / buildings.bh.clip(lower=1e-6)
buildings["label"]  = buildings.subtype.map(NAME_TO_IDX)     # NaN for un-classified

scene_meta = scene_meta.merge(scenes[["scene_id", "disaster", "origin", "split"]],
                              on="scene_id", how="left")

print(f"\n{len(buildings):,} building polygons across "
      f"{buildings.scene_id.nunique():,} post-disaster scenes")
display(buildings.head(3))

In [ ]:
# ---------------------------------------------------------------- cleaning rules 4.4 + 4.5
# Re-derived here so this notebook is standalone. Identical thresholds to 01_eda.ipynb, which
# is where each one is justified and its per-split cost is reported.
n_all = len(buildings)
valid = buildings[buildings.label.notna()].copy()             # 4.4a: drop un-classified
valid = valid[(valid.bw > 0) & (valid.bh > 0)]                # 4.4b: drop degenerate bboxes
too_small = (valid.bw < MIN_BBOX_PX) | (valid.bh < MIN_BBOX_PX)
valid = valid[~too_small].copy()                              # 4.5: drop sub-8px footprints

print(f"polygons: {n_all:,} raw -> {len(valid):,} usable "
      f"({100 * len(valid) / max(n_all, 1):.1f}% kept)")
display(valid.groupby("split").size().reindex(SPLITS).rename("usable polygons").to_frame())

## 3 · The split is inherited, not invented

The split comes from `scripts/prepare_splits.py` on `Bhuvanesh-branch` and is reused verbatim.
Re-deriving it here would risk a different train/val draw than the one the from-scratch model uses,
and the two models must be compared on identical data or the comparison means nothing.

Design, for the record:

| split | rule | seed dependence |
|---|---|---|
| `train` | tier1+tier3, non-fire, 88.9% of the pool, stratified by event | seed 42 |
| `val` | tier1+tier3, non-fire, 11.1% of the pool, stratified by event | seed 42 |
| `test_id` | official `test` + `hold` directories, non-fire | none — defined by the data |
| `test_ood` | every wildfire scene, any origin | none — defined by the data |

Two properties worth defending in the report:

- **`test_id` is drawn from the official evaluation directories, not from tier1/tier3.** Those scenes
  were never in any training pool and, unlike `val`, were never used for tuning. That makes them an
  honest reference against which to read the OOD number.
- **Assignment is at scene level**, which is what guarantees the pre and post image of a location
  stay together and that no building can straddle two splits.

### 3.1 Leakage checks, at building level

In [ ]:
# ---------------------------------------------------------------- leakage checks
b_split = valid.groupby("scene_id").split.nunique()
assert (b_split == 1).all(), "a scene appears in more than one split"

fire_in_train = valid[valid.split.isin(["train", "val"]) & valid.disaster.isin(FIRE_EVENTS)]
assert len(fire_in_train) == 0, "wildfire buildings leaked into train/val"

ood_events   = set(valid.loc[valid.split == "test_ood", "disaster"])
train_events = set(valid.loc[valid.split.isin(["train", "val"]), "disaster"])
print("events in train/val :", sorted(train_events))
print("events in test_ood  :", sorted(ood_events))
print("overlap (must be empty):", sorted(train_events & ood_events))
assert not (train_events & ood_events)

# scene_id carries the event name, so a scene cannot silently move between events either
assert (valid.scene_id.str.rsplit("_", n=1).str[0] == valid.disaster).all(), \
    "scene_id does not agree with its disaster label"
print("\nscene_id <-> disaster agreement: OK")

In [ ]:
print("usable polygons per split x class")
ct = pd.crosstab(valid.split, valid.subtype).reindex(index=SPLITS, columns=DAMAGE_CLASSES,
                                                     fill_value=0)
ct = ct.assign(total=ct.sum(axis=1))
display(ct)
ct.to_csv(RESULTS / "usable_polygons_per_split.csv")

pct = ct[DAMAGE_CLASSES].div(ct.total.clip(lower=1), axis=0) * 100
print("row-normalised % - this is the prior each split presents to the model")
display(pct.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
pct.plot(kind="bar", stacked=True, ax=ax)
ax.set(title="Damage-class composition per split, after cleaning (%)",
       ylabel="% of usable polygons", xlabel="")
ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
fig.savefig(RESULTS / "composition_per_split_after_cleaning.png", dpi=140, bbox_inches="tight")
plt.show()

## 4 · Scene sampling under a budget

All 11,034 scenes will not fit a single training session, so scenes are sampled. Three rules keep it
defensible:

1. **Sample scenes, never buildings** — the split is defined at scene level, so the sampling unit
   must match it.
2. **Stratify by event** so no event disappears.
3. **Require ≥ `MIN_BUILDINGS_PER_SCENE` usable polygons**, justified by the 54.1% empty-scene rate
   measured in EDA 3.3 and applied identically to all four splits.

On the reference run, eligible after filtering: 2,393 train / 293 val / 910 test_id / 1,990 test_ood.

*Reading the mix table below:* it compares each event's share of the **whole** eligible pool against
its share of the **whole** selection, so fire events look under-represented simply because
`test_ood` draws 400 of 1,990 (20%) while `train` draws 800 of 2,393 (33%). The stratification that
matters happens **within** each split, and that is preserved.

In [ ]:
set_seed()
usable_counts = valid.groupby("scene_id").size().rename("n_usable")
pool = scenes.merge(usable_counts, left_on="scene_id", right_index=True, how="inner")
pool = pool[pool.n_usable >= MIN_BUILDINGS_PER_SCENE]

print(f"scenes eligible for extraction (files ok, >= {MIN_BUILDINGS_PER_SCENE} usable buildings):")
elig = (pool.split.value_counts().reindex(SPLITS).fillna(0).astype(int)
        .rename("eligible").to_frame().assign(budget=[SCENE_BUDGET[s] for s in SPLITS]))
elig["will_take"] = np.minimum(elig.eligible, elig.budget)
display(elig)

In [ ]:
def sample_scenes(group, n):
    """Stratified-by-event sample of `n` scenes, proportional to each event's share."""
    if len(group) <= n:
        return group
    share = group.disaster.value_counts(normalize=True)
    take = (share * n).round().astype(int).clip(lower=1)
    out = [g.sample(min(len(g), take.get(ev, 1)), random_state=SEED)
           for ev, g in group.groupby("disaster")]
    out = pd.concat(out)
    if len(out) > n:                                    # trim the rounding overshoot
        out = out.sample(n, random_state=SEED)
    return out


set_seed()
selected = pd.concat([sample_scenes(g, SCENE_BUDGET[sp]) for sp, g in pool.groupby("split")],
                     ignore_index=True)

print("selected scenes:")
display(selected.split.value_counts().reindex(SPLITS).fillna(0).astype(int)
        .rename("scenes").to_frame())

cmp = pd.DataFrame({
    "eligible_%": (100 * pool.groupby("disaster").size() / len(pool)).round(1),
    "selected_%": (100 * selected.groupby("disaster").size() / len(selected)).round(1),
}).fillna(0)
print("event mix preserved? (selected % vs eligible %)")
display(cmp)
cmp.to_csv(RESULTS / "event_mix_eligible_vs_selected.csv")

In [ ]:
# ---------------------------------------------------------------- within-split stratification
# The table above compares each event against the WHOLE pool, which is the wrong denominator
# for judging stratification. This one uses the right denominator: within each split.
within = []
for sp in SPLITS:
    p, s = pool[pool.split == sp], selected[selected.split == sp]
    if not len(p):
        continue
    within.append(pd.DataFrame({
        "split": sp,
        "eligible_%": (100 * p.groupby("disaster").size() / len(p)).round(1),
        "selected_%": (100 * s.groupby("disaster").size() / max(len(s), 1)).round(1),
    }).fillna(0))
within = pd.concat(within).reset_index().rename(columns={"index": "disaster"})
within["abs_drift_pp"] = (within["selected_%"] - within["eligible_%"]).abs().round(1)
display(within.sort_values("abs_drift_pp", ascending=False).head(20))
print("worst within-split drift:", within.abs_drift_pp.max(), "percentage points")
within.to_csv(RESULTS / "event_mix_within_split.csv", index=False)

## 5 · Export

`results/02_data_split/selected_scenes.csv` is the hand-off to patch extraction: one row per selected scene
with the two resolved image paths, the split, and the number of usable buildings it contributes.
Small, reviewable, and worth committing.

`work/split/patch_manifest.csv` is the row-level plan — one row per building that will become a
training sample — so the exact patch count and class balance are known before a single PNG is
decoded. Tens of MB and regenerable in seconds, so it lives in `work/` and is gitignored.

In [ ]:
sel_out = selected[["scene_id", "disaster", "origin", "split", "n_usable",
                    "img_pre", "img_post", "lab_post"]].sort_values(["split", "scene_id"])
sel_out.to_csv(RESULTS / "selected_scenes.csv", index=False)

patch_manifest = (valid.merge(selected[["scene_id", "img_pre", "img_post"]], on="scene_id")
                       .reset_index(drop=True))
patch_manifest["row"] = np.arange(len(patch_manifest))

# One row per building, absolute paths included: tens of MB, and regenerable from the two files
# above in seconds. That makes it derived data -> work/, not results/.
SPLIT_WORK = WORK / "split"; SPLIT_WORK.mkdir(parents=True, exist_ok=True)
MANIFEST = SPLIT_WORK / "patch_manifest.csv"
patch_manifest.to_csv(MANIFEST, index=False)

N = len(patch_manifest)
print(f"selected scenes    : {len(sel_out):,}")
print(f"patches to extract : {N:,}")
print(f"array size on disk : {2 * N * PATCH_SIZE * PATCH_SIZE * 3 / 1e9:.2f} GB "
      f"({2 * PATCH_SIZE * PATCH_SIZE * 3 / 1024:.1f} KB per pre/post pair)")

mix = (pd.crosstab(patch_manifest.split, patch_manifest.subtype)
         .reindex(index=SPLITS, columns=DAMAGE_CLASSES, fill_value=0))
display(mix.assign(total=mix.sum(axis=1)))
mix.to_csv(RESULTS / "patch_class_balance.csv")

print("\nwrote:")
print("  ", RESULTS / "selected_scenes.csv")
print("  ", RESULTS / "patch_class_balance.csv")
print("  ", MANIFEST, "(derived, gitignored)")

In [ ]:
# ---------------------------------------------------------------- class weights, from TRAIN only
# The one number downstream training needs from this notebook. Computed here rather than in the
# model notebook so both models (scratch and pretrained) provably use the same weights.
train_counts = (patch_manifest[patch_manifest.split == "train"].label.value_counts()
                .reindex(range(NUM_CLASSES)).fillna(0))

# Inverse frequency, the sklearn "balanced" convention: n_samples / (n_classes * count).
# Chosen over resampling: oversampling would repeat the same few destroyed buildings many times
# per epoch and invite memorisation, undersampling would discard most of the majority signal.
# Weighting keeps every sample once and changes only its gradient contribution.
w = (train_counts.sum() / (NUM_CLASSES * train_counts.clip(lower=1))).round(3)
weights = pd.DataFrame({"class": DAMAGE_CLASSES, "train_patches": train_counts.astype(int).values,
                        "weight": w.values})
display(weights)
weights.to_csv(RESULTS / "train_class_weights.csv", index=False)

## 6 · What this section established

- **The split is scene-level and inherited**, so no building can straddle two splits and pre/post
  tiles of a location can never separate. Both assertions are re-checked here after the path
  rewrite, because the split is the one thing that can silently invalidate every downstream number.
- **Zero wildfire leakage into `train`/`val`** — the event sets are disjoint, which is what makes
  `test_ood` a genuine zero-shot measurement rather than a harder validation set.
- **`test_id` comes from the official `test` + `hold` directories**, never tuned on, so the ID→OOD
  gap is read against an honest in-distribution reference rather than against `val`.
- **Scene sampling is stratified within each split** and filtered at
  `MIN_BUILDINGS_PER_SCENE = 5`, applied identically to all four splits — so it cannot tilt the
  comparison the project is built around.

On the reference run this produced **135,262 patch pairs from 1,698 scenes** —
72,916 train / 20,446 val / 27,273 test_id / 14,627 test_ood.

**Next:** the model notebooks consume `work/split/patch_manifest.csv`.